# M2BERT Fine-tuning for Pitch Correction

Compare pretrained **M2BERT** (ModernBERT-base backbone, CP compound-token representation)
against the custom Transformer **v5** on the OLiMPiC pitch-correction task.

**Setup**: freeze all but last 2 transformer layers (10, 11) + new 52-class head.  
**Budget**: ≤ 3 epochs **or** ≤ 10 minutes.  
**Baseline**: v5 dev top-1 = 83/499 (16.63%), top-5 = 261/499 (52.30%).


In [ ]:
import sys, json, re, random, time, pickle
from pathlib import Path
from fractions import Fraction as _Frac
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ModernBertConfig, ModernBertModel


In [ ]:
ML_DIR       = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml')
PROCESSED    = ML_DIR / 'data' / 'processed'
PRETRAINED   = ML_DIR / 'data' / 'pretrained' / 'm2bert'
M2_RUN_DIR   = ML_DIR / 'data' / 'runs' / 'm2bert_finetune'
DATASET_ROOT = ML_DIR / 'dataset' / 'olimpic-1.0-synthetic'
M2_RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'DEVICE: {DEVICE}')

# ── CP vocab (from M2BERT pretrained) ──────────────────────────
with open(PRETRAINED / 'CP.pkl', 'rb') as f:
    _cp = pickle.load(f)

# _cp[0] = {attr_name: {token_str: id}}
# _cp[1] = {attr_idx: {id: token_str}}
CP_TOK2ID = _cp[0]
print('CP attribute sizes:', {k: len(v) for k, v in CP_TOK2ID.items()})

BAR_PAD    = CP_TOK2ID['Bar']['Bar <PAD>']           # 2
BAR_MASK   = CP_TOK2ID['Bar']['Bar <MASK>']          # 3
POS_PAD    = CP_TOK2ID['Position']['Position <PAD>'] # 16
PITCH_PAD  = CP_TOK2ID['Pitch']['Pitch <PAD>']       # 86
PITCH_MASK = CP_TOK2ID['Pitch']['Pitch <MASK>']      # 87
DUR_PAD    = CP_TOK2ID['Duration']['Duration <PAD>'] # 64

CP_PAD = [BAR_PAD, POS_PAD, PITCH_PAD, DUR_PAD]
print(f'CP_PAD: {CP_PAD}   PITCH_MASK: {PITCH_MASK}')


In [ ]:
PITCH_RE = re.compile(r'^[A-G][0-9]$')

def _lmx_path(sid):
    parts = sid.split('/')
    return DATASET_ROOT / 'samples' / parts[1] / (parts[2] + '.lmx')

# 52-class pitch mapping — identical to main notebook's build_pitch_classes()
pitches = set()
for sid in (DATASET_ROOT / 'samples.train.txt').read_text().split():
    p = _lmx_path(sid)
    if p.exists():
        pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
pitch_classes = {p: i for i, p in enumerate(sorted(pitches))}
id2pitch      = {i: p for p, i in pitch_classes.items()}
N_CLASSES     = len(pitch_classes)
print(f'Pitch classes: {N_CLASSES}  (classes: {sorted(pitch_classes)[:5]} ... {sorted(pitch_classes)[-3:]})')

with open(PROCESSED / 'correction_train_10k.jsonl') as f:
    train_raw = [json.loads(l) for l in f]
with open(PROCESSED / 'correction_dev_v1.jsonl') as f:
    dev_raw = [json.loads(l) for l in f]
print(f'Train: {len(train_raw)}  Dev: {len(dev_raw)}')


## LMX → CP Conversion

Each example's full token sequence is parsed into per-note dicts by `parse_lmx_notes`.
For each note we compute four CP indices:

| Attribute | Formula |
|-----------|---------|
| Bar       | 0 (Bar New) if note starts a new measure vs previous note, else 1 (Bar Continue) |
| Position  | `min(15, int(onset_in_measure_whole_notes × 16))` |
| Pitch     | `MIDI - 22` (0–85); 87 = PITCH_MASK for error note; 86 = PITCH_PAD for rest |
| Duration  | `min(63, int(duration_whole_notes × 32))`; 64 = DUR_PAD if unknown |

The full note sequence (no windowing needed; max 216 notes across all examples) is used.


In [ ]:
# Note-Level LMX Parser — copied verbatim from correction_experiments.ipynb cell 41.
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}
_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})
_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})
_SKIP = frozenset({'time'})
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')

def _parse_dur(tokens, i):
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i

def parse_lmx_notes(tokens, sample_id=''):
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)
    while i < n:
        tok = tokens[i]
        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1; continue
        if tok in _SKIP or tok.startswith('key:'):
            i += 1; continue
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats {tok!r}')
            i += 1; continue
        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type {tok!r}')
            i += 1; continue
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'): i += 1
            continue
        if tok in _DECORATION:
            i += 1; continue
        span_start = i
        is_grace = is_chord = False
        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]
        if tok == 'grace':
            is_grace = True; i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]
        if tok == 'chord':
            is_chord = True; i += 1
            if i >= n: break
            tok = tokens[i]
        pitch = None; is_rest = False; pitch_token_position = None
        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i; i += 1
        elif tok == 'rest':
            is_rest = True; i += 1
        elif tok == 'backup':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset = max(_Frac(0), current_onset - dur)
            else: warnings.append(f'backup no dur @ {i}')
            continue
        elif tok == 'forward':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset += dur
            else: warnings.append(f'forward no dur @ {i}')
            continue
        else:
            warnings.append(f'unexpected {tok!r} @ {i}'); i += 1; continue
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]; i += 1
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(f'rest:measure unknown dur @ {sample_id}')
                i += 1
        if i < n and tokens[i] in _ACCIDENTAL: i += 1
        if i < n and tokens[i].startswith('stem:'): i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]; i += 1
        while i < n and tokens[i] in _DECORATION: i += 1
        onset = chord_leader_onset if is_chord else current_onset
        if is_chord: grp = chord_group_id
        else:
            chord_group_id += 1; grp = chord_group_id
            chord_leader_onset = current_onset
        notes.append({'measure': measure_num, 'voice': current_voice, 'staff': current_staff,
                      'onset': onset, 'duration': duration, 'pitch': pitch,
                      'is_rest': is_rest, 'is_grace': is_grace,
                      'chord_group': grp, 'pitch_token_position': pitch_token_position,
                      'token_span': (span_start, i)})
        if not is_chord and not is_grace and duration is not None:
            current_onset += duration
    return notes, warnings

print('parse_lmx_notes defined')


In [ ]:
_SEMITONES = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _pitch_to_cp_idx(tok):
    midi = (int(tok[1]) + 1) * 12 + _SEMITONES[tok[0]]
    idx  = midi - 22
    return idx if 0 <= idx <= 85 else None

def lmx_to_cp_example(ex, pitch_classes):
    """
    Convert one JSONL example to (cp_ids, error_note_idx, target_class).
    cp_ids: list[list[int]] of shape [n_notes, 4] = [bar, pos, pitch, dur].
    Error note pitch is replaced with PITCH_MASK (87).
    Returns None only if error note position not found (0 in practice).
    """
    target = ex['target_token']
    if target not in pitch_classes:
        return None

    notes, _ = parse_lmx_notes(ex['input_tokens'], ex.get('sample_id', ''))
    err_pos   = ex['error_position']
    err_idx   = next((i for i, n in enumerate(notes)
                      if n['pitch_token_position'] == err_pos), None)
    if err_idx is None:
        return None

    cp_ids    = []
    prev_meas = -1
    for i, note in enumerate(notes):
        bar   = 0 if note['measure'] != prev_meas else 1
        prev_meas = note['measure']
        pos   = min(15, int(float(note['onset']) * 16))
        if i == err_idx:
            pitch = PITCH_MASK
        elif note['pitch'] and PITCH_RE.match(note['pitch']):
            p = _pitch_to_cp_idx(note['pitch'])
            pitch = p if p is not None else PITCH_PAD
        else:
            pitch = PITCH_PAD   # rest
        if note['duration'] is not None:
            dur = min(63, int(float(note['duration']) * 32))
        else:
            dur = DUR_PAD
        cp_ids.append([bar, pos, pitch, dur])

    return cp_ids, err_idx, pitch_classes[target]


In [ ]:
def build_cp_examples(raw_list, pitch_classes, split_name):
    examples, skipped = [], 0
    for ex in raw_list:
        res = lmx_to_cp_example(ex, pitch_classes)
        if res is None:
            skipped += 1; continue
        cp_ids, err_idx, tgt = res
        examples.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                         'target_class': tgt, 'sample_id': ex['sample_id']})
    lens = [len(e['cp_ids']) for e in examples]
    print(f'{split_name}: {len(examples)} ok, {skipped} skipped  '
          f'seq_len min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}')
    return examples

print('Building CP examples...')
t0 = time.time()
train_cp = build_cp_examples(train_raw, pitch_classes, 'train')
dev_cp   = build_cp_examples(dev_raw,   pitch_classes, 'dev')
print(f'Done in {time.time()-t0:.1f}s')


## M2BERT Model

- **Backbone** (`MidiModernBERT`): 4 × Embedding(vocab_size, 256) → Linear(1024, 768)
  → ModernBERT-base (12 layers, 768 hidden). Loaded from `model_best.ckpt` with `strict=True`.
- **Head**: Dropout(0.1) → Linear(768, 52). Takes hidden state at the masked note position.
- **Frozen**: all backbone except layers 10, 11 (last 2 transformer blocks).


In [ ]:
class LUT(nn.Module):
    """Lookup-table wrapper matching M2BERT checkpoint key `word_emb.N.lut.weight`."""
    def __init__(self, n, d, pad=None):
        super().__init__()
        self.lut = nn.Embedding(n, d, padding_idx=pad)
    def forward(self, x):
        return self.lut(x)


class MidiModernBERT(nn.Module):
    """M2BERT backbone: CP embeddings → ModernBERT-12L."""
    def __init__(self):
        super().__init__()
        self.word_emb = nn.ModuleList([
            LUT(4,  256, pad=BAR_PAD),    # Bar
            LUT(18, 256, pad=POS_PAD),    # Position
            LUT(88, 256, pad=PITCH_PAD),  # Pitch
            LUT(66, 256, pad=DUR_PAD),    # Duration
        ])
        self.in_linear = nn.Linear(1024, 768)
        _cfg = ModernBertConfig(
            vocab_size=10, hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=1152,
            max_position_embeddings=4096, pad_token_id=0,
        )
        self.note_bert1 = ModernBertModel(_cfg)

    def forward(self, cp_ids, attention_mask=None):
        # cp_ids: [B, S, 4]
        embs = torch.cat([self.word_emb[i](cp_ids[:, :, i]) for i in range(4)], dim=-1)
        embs = self.in_linear(embs)           # [B, S, 768]
        out  = self.note_bert1(inputs_embeds=embs, attention_mask=attention_mask)
        return out.last_hidden_state          # [B, S, 768]


class M2BERTPitchCorrection(nn.Module):
    """M2BERT backbone + classification head for pitch correction."""
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_classes))

    def forward(self, cp_ids, error_note_idx, attention_mask=None):
        hidden  = self.backbone(cp_ids, attention_mask)                   # [B, S, 768]
        err_h   = hidden[torch.arange(hidden.size(0)), error_note_idx]    # [B, 768]
        return self.classifier(err_h)                                     # [B, N_CLASSES]


In [ ]:
backbone = MidiModernBERT()
_ckpt    = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu', weights_only=False)
backbone.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone: strict=True load OK  '
      f'(pretrained epoch {_ckpt["epoch"]}, best_acc={_ckpt["best_acc"]:.4f})')

model = M2BERTPitchCorrection(backbone, N_CLASSES)
model.to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_p:,}  N_CLASSES: {N_CLASSES}')


In [ ]:
# Freeze all backbone parameters
for p in model.parameters():
    p.requires_grad_(False)

# Unfreeze last 2 transformer layers (10 & 11) and classifier head
for layer_idx in (10, 11):
    for p in model.backbone.note_bert1.layers[layer_idx].parameters():
        p.requires_grad_(True)
for p in model.classifier.parameters():
    p.requires_grad_(True)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frz   = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable: {n_train:,}  Frozen: {n_frz:,}')

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
loss_fn   = nn.CrossEntropyLoss()


In [ ]:
class CPDataset(Dataset):
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

_pad_t = torch.tensor(CP_PAD, dtype=torch.long)

def collate_cp(batch):
    max_len    = max(len(e['cp_ids']) for e in batch)
    B          = len(batch)
    cp_ids_t   = _pad_t.unsqueeze(0).expand(max_len, -1).unsqueeze(0).expand(B, -1, -1).clone()
    attn_mask  = torch.zeros(B, max_len, dtype=torch.long)
    err_idx_t  = torch.zeros(B, dtype=torch.long)
    tgt_cls_t  = torch.zeros(B, dtype=torch.long)
    for i, e in enumerate(batch):
        L = len(e['cp_ids'])
        cp_ids_t[i, :L] = torch.tensor(e['cp_ids'], dtype=torch.long)
        attn_mask[i, :L] = 1
        err_idx_t[i]     = e['error_note_idx']
        tgt_cls_t[i]     = e['target_class']
    return cp_ids_t, err_idx_t, tgt_cls_t, attn_mask

# Benchmark single batch forward pass
_bmark_loader = DataLoader(CPDataset(train_cp[:64]), batch_size=64,
                            shuffle=False, collate_fn=collate_cp)
_cp, _ei, _tc, _am = next(iter(_bmark_loader))
model.eval()
with torch.no_grad():
    t0 = time.time()
    _ = model(_cp.to(DEVICE), _ei.to(DEVICE), _am.to(DEVICE))
    torch.mps.synchronize() if DEVICE.type == 'mps' else None
    print(f'Single batch (B=64): {(time.time()-t0)*1000:.1f}ms  '
          f'seq_len≈{_am.sum(1).float().mean().item():.0f}')
del _bmark_loader, _cp, _ei, _tc, _am


## Training ⚠️ TRAINS

Budget: ≤ 3 epochs **or** ≤ 10 minutes. Best checkpoint saved by dev top-1.


In [ ]:
# ⚠️ TRAINS — fine-tune M2BERT on pitch correction.
# Requires: all cells above. Saves best checkpoint to M2_RUN_DIR.

MAX_EPOCHS  = 3
MAX_SECONDS = 600          # 10-minute budget
BATCH_SIZE  = 32

train_loader = DataLoader(CPDataset(train_cp), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_cp, num_workers=0)
dev_loader   = DataLoader(CPDataset(dev_cp),   batch_size=64, shuffle=False,
                          collate_fn=collate_cp, num_workers=0)


def _eval_top1(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            logits  = model(cp_ids.to(device), err_idx.to(device), attn.to(device))
            correct += (logits.argmax(1) == tgt.to(device)).sum().item()
            total   += tgt.size(0)
    return correct / total


t_start       = time.time()
metrics       = []
best_dev_top1 = 0.0
best_ckpt_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    if time.time() - t_start >= MAX_SECONDS:
        print(f'Budget reached before epoch {epoch}'); break
    model.train()
    total_loss, n_batches = 0.0, 0
    for cp_ids, err_idx, tgt, attn in train_loader:
        if time.time() - t_start >= MAX_SECONDS:
            print(f'Budget reached mid-epoch {epoch}'); break
        optimizer.zero_grad()
        logits = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
        loss   = loss_fn(logits, tgt.to(DEVICE))
        loss.backward()
        optimizer.step()
        total_loss += loss.item(); n_batches += 1
    train_loss = total_loss / max(n_batches, 1)
    dev_top1   = _eval_top1(model, dev_loader, DEVICE)
    elapsed    = time.time() - t_start
    row = {'epoch': epoch, 'train_loss': round(train_loss, 4),
           'dev_top1': round(dev_top1, 4), 'elapsed_s': round(elapsed, 1)}
    metrics.append(row)
    star = '  *best*' if dev_top1 > best_dev_top1 else ''
    print(f'Epoch {epoch}: loss={train_loss:.4f}  dev_top1={dev_top1:.4f} '
          f'({dev_top1*len(dev_cp):.0f}/{len(dev_cp)})  t={elapsed:.0f}s{star}')
    if dev_top1 > best_dev_top1:
        best_dev_top1   = dev_top1
        best_ckpt_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Save best checkpoint
if best_ckpt_state is not None:
    torch.save({'model_state_dict': best_ckpt_state, 'best_dev_top1': best_dev_top1,
                'metrics': metrics, 'n_classes': N_CLASSES,
                'pitch_classes': pitch_classes},
               M2_RUN_DIR / 'best_checkpoint.pt')
with open(M2_RUN_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nBest dev top-1: {best_dev_top1:.4f} ({best_dev_top1*len(dev_cp):.0f}/{len(dev_cp)})')
print(f'Saved to {M2_RUN_DIR}')


## Results — M2BERT vs v5 📊

Load best checkpoint, evaluate on all 499 dev examples, compare with v5.


In [ ]:
# 📊 Evaluate M2BERT on dev + compare with v5.
# Requires: best_checkpoint.pt saved by the TRAINS cell.

# Load best checkpoint
_saved = torch.load(M2_RUN_DIR / 'best_checkpoint.pt', map_location='cpu', weights_only=False)
model.load_state_dict(_saved['model_state_dict'])
model.to(DEVICE)
model.eval()
print(f'Loaded checkpoint: best_dev_top1={_saved["best_dev_top1"]:.4f}')

# Full dev evaluation with per-example predictions
all_preds = []
with torch.inference_mode():
    for cp_ids, err_idx, tgt, attn in dev_loader:
        logits   = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
        probs    = logits.softmax(dim=-1).cpu()
        top5_ids = probs.topk(5, dim=-1).indices.tolist()
        for b in range(tgt.size(0)):
            pred_id  = top5_ids[b][0]
            tgt_id   = tgt[b].item()
            all_preds.append({
                'sample_id':      dev_cp[len(all_preds)]['sample_id'],
                'target_token':   id2pitch[tgt_id],
                'predicted_token': id2pitch[pred_id],
                'correct':        (pred_id == tgt_id),
                'top5_candidates': [id2pitch[x] for x in top5_ids[b]],
            })

with open(M2_RUN_DIR / 'dev_predictions.json', 'w') as f:
    json.dump(all_preds, f, indent=2)

m2_t1 = sum(1 for p in all_preds if p['correct'])
m2_t5 = sum(1 for p in all_preds if p['target_token'] in p['top5_candidates'])
n_dev = len(all_preds)
print(f'M2BERT  dev: top-1 = {m2_t1}/{n_dev} ({m2_t1/n_dev:.2%})  '
      f'top-5 = {m2_t5}/{n_dev} ({m2_t5/n_dev:.2%})')

# ── Load v5 for comparison ─────────────────────────────────────
V5_DIR     = ML_DIR / 'data' / 'runs' / 'v5_voice_conditioned'
_v5_preds  = json.load(open(V5_DIR / 'dev_predictions.json'))
v5_t1 = sum(1 for p in _v5_preds if p['correct'])
v5_t5 = sum(1 for p in _v5_preds if p['target_token'] in p['top5_candidates'])
print(f'v5      dev: top-1 = {v5_t1}/{n_dev} ({v5_t1/n_dev:.2%})  '
      f'top-5 = {v5_t5}/{n_dev} ({v5_t5/n_dev:.2%})')

# ── Venn: overlap ──────────────────────────────────────────────
m2_ids = {p['sample_id'] for p in all_preds if p['correct']}
v5_ids = {p['sample_id'] for p in _v5_preds if p['correct']}
both  = len(m2_ids & v5_ids)
only_m2 = len(m2_ids - v5_ids)
only_v5 = len(v5_ids - m2_ids)
print(f'\nVenn overlap  both={both}  only_M2BERT={only_m2}  only_v5={only_v5}')

# ── Summary table ─────────────────────────────────────────────
_met = json.load(open(M2_RUN_DIR / 'metrics.json'))
n_ep = len(_met)
partial = ' (partial)' if n_ep < 3 else ''
print(f'\n{"Model":<22} {"Top-1":>10} {"Top-5":>10} {"Epochs":>8}')
print('─' * 55)
print(f'{"v5 (voice-conditioned)":<22} '
      f'{v5_t1}/{n_dev} ({v5_t1/n_dev:.2%}) {" ":>4} '
      f'{v5_t5}/{n_dev} ({v5_t5/n_dev:.2%}) '
      f'{"10":>8}')
print(f'{"M2BERT fine-tuned":<22} '
      f'{m2_t1}/{n_dev} ({m2_t1/n_dev:.2%}) {" ":>4} '
      f'{m2_t5}/{n_dev} ({m2_t5/n_dev:.2%}) '
      f'{str(n_ep)+partial:>8}')
